In [ ]:
import * as tslab from "tslab";
import { readFileSync } from "fs";

const css = readFileSync("../style.css", "utf-8");
tslab.display.html(`<style>${css}</style>`);

# A Parser for First Order Logic 

In [ ]:
const lexSpec = /([ ,:\t]+)|([a-z][A-Za-z0-9]*)|([A-Z][A-Za-z0-9]*)|([⊤⊥∧∨¬→↔⊕∀∃()])/g;

In [ ]:
function tokenize(s: string): string[] {
    const tokenArray = Array.from(s.matchAll(lexSpec));
    const result: string[] = [];
    for (const match of tokenArray) {
        const ws = match[1];
        const identifier = match[2];
        const funcSymbol = match[3];
        const operator = match[4];
        if (ws) {
            continue;
        }
        if (identifier) {
            result.push(identifier);
        }
        if (funcSymbol) {
            result.push(funcSymbol);
        }
        if (operator) {
            result.push(operator);
        }
    }
    return result;
}

Variable names start with a lower case letter.

In [ ]:
function isVariable(s: string, Variables: Set<string> | null): boolean {
    // Check, whether the string s can be interpreted as a variable. 
    const varRegex = /^[a-z][A-Za-z0-9]*$/;
    if (Variables === null) {
        return varRegex.test(s);
    } else {
        if (varRegex.test(s)) {
            if (Variables.has(s)) {
                return true;
            } else {
                console.error(`Syntax error: ${s} is not declared as a variable!`);
                throw new SyntaxError(`Variable "${s}" not declared`);
            }
        }
    }
    return false;
}

Function names start with an upper case letter. 

In [ ]:
function isFunction(s: string): boolean {
    // Check whether the string s can be interpreted as a function symbol.
    // Syntactically, we do not distinguish between function symbols and predicate symbols.
    const funcRegex = /^[A-Z][A-Za-z0-9]*$/;
    return funcRegex.test(s);
}

In [ ]:
type Formula = string | [string, ...Formula[]];

In [ ]:
class LogicParser {
    // This class implements the shunting yard algorithm to parse formulas from
    // first order logic.  The strings that represent formulas are transformed
    // into nested tuples that are interpreted as Formula representing the 
    // formulae.
    
    private _tokens: string[];
    private _operators: string[];
    private _arguments: Formula[];
    private _variables?: Set<string> | null;
    private _input: string;

    constructor(s: string, Variables?: Set<string> | null) {
        this._tokens     = tokenize(s).reverse();
        this._operators  = [];
        this._arguments  = [];
        this._variables  = Variables ?? null;
        this._input      = s;
    }

    parse(): Formula {
        // Parse the token list and return a Formula that is represented as a
        // nested array.
        while (this._tokens.length !== 0) {
            const nextOp = this._tokens.pop()!;
            if (isVariable(nextOp, this._variables)) {
                this._arguments.push(nextOp);
                continue;
            }
            if (isFunction(nextOp)) {
                this._operators.push(nextOp);
                this._arguments.push("(");
                continue;
            }
            if (nextOp === "⊤" || nextOp === "⊥") {
                this._operators.push(nextOp);
                continue;
            }
            if (this._operators.length === 0 || nextOp === "(") {
                this._operators.push(nextOp);
                continue;
            }
            const stackOp = this._operators[this._operators.length - 1];
            if (stackOp === "(" && nextOp === ")") {
                this._operators.pop();
                if (this._operators.length > 0) {
                    const fct = this._operators[this._operators.length - 1];
                    if (isFunction(fct)) {
                        this._popAndEvaluate();
                    }
                }
            } else if (nextOp === ")" || this._evalBefore(stackOp, nextOp)) {
                this._popAndEvaluate();
                this._tokens.push(nextOp);
            } else {
                this._operators.push(nextOp);
            }
        }
        while (this._operators.length !== 0) {
            this._popAndEvaluate();
        }
        if (this._arguments.length !== 1) {
            throw new Error(`Could not parse: ${this._input}`);
        }
        return this._arguments.pop()!;
    }

    _evalBefore(stackOp: string, nextOp: string): boolean {
        // Check if the operator on top of the operator stack should be evaluated
        // before the next operator from the input array.
        if (stackOp === "(") return false;
        if (isFunction(stackOp)) return true;
        const precedences: { [key: string]: number } = {
            "↔": 1, "→": 2, "⊕": 3, "∨": 4, "∧": 5,
            "¬": 6, "∀": 7, "∃": 7, "⊤": 8, "⊥": 8,
        };
        if (precedences[stackOp] > precedences[nextOp]) {
            return true;
        } else if (precedences[stackOp] === precedences[nextOp]) {
            if ((stackOp === "∀" || stackOp === "∃") && (nextOp === "∀" || nextOp === "∃")) {
                return false;
            }
            if (stackOp === nextOp) {
                return ["∧", "∨", "⊕"].includes(stackOp);
            }
            return true;
        }
        return false;
    }

    _popAndEvaluate(): void {
        const op = this._operators.pop()!;
        if (op === "⊤" || op === "⊥") {
            this._arguments.push([op]);
            return;
        }
        if (op === "¬") {
            const arg = this._arguments.pop()!;
            this._arguments.push(["¬", arg]);
            return;
        }
        if (isFunction(op)) {
            let args: Formula[] = [];
            let arg = this._arguments.pop();
            while (arg !== "(") {
                args.unshift(arg!);
                arg = this._arguments.pop();
            }
            this._arguments.push([op, ...args]);
            return;
        }
        const rhs = this._arguments.pop()!;
        const lhs = this._arguments.pop()!;
        this._arguments.push([op, lhs, rhs]);
    }

    toString(): string {
        return `${this._tokens.toString()} ${this._arguments.toString()} ${this._operators.toString()}`;
    }
}

In [ ]:
function testParser(s: string): void {
    const p = new LogicParser(s);
    console.log('\n');
    console.log(`parsing: "${s}"`);
    console.dir(p.parse(),{depth: null});
}

In [ ]:
function runTest() {
    testParser('G(F(x,y),x)');
    testParser('P(F(x),G(z))');
    testParser('∀x:∃y:P(x,y)');
    testParser('∀x:∃y:P(x,y)→∃y:∀x:P(x,y)');
    testParser('¬∀x:(Red(x) → Happy(x))');
    testParser('∀x:∀y:(¬P(F(x),y)) ∨ ∀u:∀v:(¬P(u,G(v)))');
}
runTest();